# Parsing d'Amazon

Je réalise le parsing de mes données Amazon car ce ne sont que des csv. Je vais les transformer en parquet pour les rendre plus facilement exploitables par la suite.

Pour ce qui est des PDFs, ce n'est pas utile de les parser, je les stockerai tels quels dans un bucket S3 et je les référencerai dans ma base de données. Je pourrai les consulter à la demande, mais je n'aurai pas besoin de les analyser pour l'instant.

---

In [1]:
from pyspark.sql import SparkSession

# Initialisation de la session
spark = SparkSession.builder.appName("Ingestion").getOrCreate()

26/03/27 12:38:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Amazon Orders
---

### Historique de commandes

In [2]:
path_amazon_orders = "/opt/spark/data/raw/AMAZON/Your Orders/Your Amazon Orders"

df_orders = spark.read.csv(path_amazon_orders, header=True, inferSchema=True)

In [3]:
df_orders.printSchema()

root
 |-- ASIN: string (nullable = true)
 |-- Billing Address: string (nullable = true)
 |-- Carrier Name & Tracking Number: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Gift Message: string (nullable = true)
 |-- Gift Recipient Contact: string (nullable = true)
 |-- Gift Sender Name: string (nullable = true)
 |-- Item Serial Number: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Original Quantity: string (nullable = true)
 |-- Payment Method Type: string (nullable = true)
 |-- Product Condition: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Purchase Order Number: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Shipment Item Subtotal: string (nullable = true)
 |-- Shipment Item Subtotal Tax: string (nullable = true)
 |-- Shipment Status: string (nullable = true)
 |-- Shipping Address: string (nullable = 

In [18]:
df_orders.show(5, truncate=False)

+----------+------------------------------------------------------------+-----------------------------------+--------+-------------+----------------------+----------------+------------------+--------------------+-------------------+------------+-----------------+-------------------+-----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------+------------------------+----------------------+--------------------------+---------------+------------------------------------------------------------+---------------+---------------+------------+---------------+----------+--------------+-------------+
|ASIN      |Billing Address                                             |Carrier Name & Tracking Number     |Currency|Gift Message |Gift Recipient Contact|Gift Sender Name|Item Serial Number|Order Date          |Order ID        

In [5]:
print(f"Nombre de lignes détectées : {df_orders.count()}")

Nombre de lignes détectées : 106


In [6]:
df_clean = df_orders.toDF(*(c.replace(' ', '_') for c in df_orders.columns))

In [7]:
df_clean.write.mode("overwrite").parquet("/opt/spark/data/processed/amazon/orders")

print("Transfert terminé ! ✅")

Transfert Amazon Orders terminé ! ✅


---
## Step 2 : Returns & Refunds

In [8]:
path_amazon_orders = "/opt/spark/data/raw/AMAZON/Your Orders/Your Returns & Refunds"

df_returns = spark.read.csv(path_amazon_orders, header=True, inferSchema=True)

In [9]:
df_returns.printSchema()

root
 |-- Carrier Package ID: string (nullable = true)
 |-- Carrier Tracking ID: string (nullable = true)
 |-- Charge Status: string (nullable = true)
 |-- Contract Creation Date: string (nullable = true)
 |-- Contract ID: string (nullable = true)
 |-- Contract Unit ID: string (nullable = true)
 |-- Date of Return: string (nullable = true)
 |-- Gift Card: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Payment Methode Service Type: string (nullable = true)
 |-- Replacement Order: string (nullable = true)
 |-- Retrocharge Resolution: string (nullable = true)
 |-- Return Amount: string (nullable = true)
 |-- Return Amount Currency: string (nullable = true)
 |-- Return Authorization ID: string (nullable = true)
 |-- Return Creation Date: string (nullable = true)
 |-- Return Reason: string (nullable = true)
 |-- Return Receivable Creation Reason: string (nullable = true)
 |-- Return Receivable Initiation Date: string (nullable = true)
 |-- Return Receivable State: str

In [11]:
df_returns.show(5, truncate=False)

+------------------------+------------------------+--------------+----------------------+------------------------------------+-------------------------------------------------------------------------+--------------------+--------------+------------------------+----------------------------+-----------------+----------------------+-------------+----------------------+-----------------------+--------------------+--------------------------------------+---------------------------------+---------------------------------+-----------------------+-----------------------------+-----------------+----------------------------+
|Carrier Package ID      |Carrier Tracking ID     |Charge Status |Contract Creation Date|Contract ID                         |Contract Unit ID                                                         |Date of Return      |Gift Card     |Order ID                |Payment Methode Service Type|Replacement Order|Retrocharge Resolution|Return Amount|Return Amount Currency|Return Aut

In [12]:
df_clean = df_returns.toDF(*(c.replace(' ', '_') for c in df_returns.columns))

In [13]:
df_clean.write.mode("overwrite").parquet("/opt/spark/data/processed/amazon/returns_refunds")

print("Transfert terminé ! ✅")

Transfert terminé ! ✅
